# EmpathRAG — FAISS Index Builder
**Run on Colab Pro with A100 GPU**

Encodes Reddit Mental Health corpus into 768-dim vectors and builds a FAISS index + SQLite sidecar.

- Estimated time on A100: 30–60 minutes
- Output: `faiss_flat.index` + `metadata.db` saved to Google Drive
- **Do not run on local RTX 3060 laptop** — would take 6–12 hours

### Before running:
1. Upload `data/raw/reddit_mental_health/` (108 CSVs, ~3.1GB) to Google Drive at `My Drive/empathrag/data/raw/reddit_mental_health/`
2. Set Runtime → Change runtime type → **A100 GPU**
3. Run all cells in order

In [ ]:
# Cell 1: Install dependencies
!pip install sentence-transformers faiss-cpu tqdm transformers -q

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

BASE       = "/content/drive/MyDrive/empathrag"
REDDIT_DIR = f"{BASE}/data/raw/reddit_mental_health"
INDEX_PATH = f"{BASE}/data/indexes/faiss_flat.index"
DB_PATH    = f"{BASE}/data/indexes/metadata.db"
CKPT_PATH  = f"{BASE}/data/indexes/embeddings_checkpoint.npy"

import os
os.makedirs(f"{BASE}/data/indexes", exist_ok=True)
print("Drive mounted. Paths configured.")
print(f"Reddit dir exists: {os.path.exists(REDDIT_DIR)}")
files = [f for f in os.listdir(REDDIT_DIR) if f.endswith('.csv')] if os.path.exists(REDDIT_DIR) else []
print(f"CSV files found: {len(files)}")

In [ ]:
# Cell 3: Helper functions
import re

def clean_text(text: str) -> str:
    text = re.sub(r"u/\w+", "", text)
    text = re.sub(r"r/\w+", "", text)
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\[deleted\]|\[removed\]", "", text)
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def chunk_text(text, tokenizer, chunk_size=256, stride=32, max_chunks=8):
    tokens = tokenizer.encode(text)
    if len(tokens) < 64:
        return [text]
    chunks = []
    start = 0
    while start < len(tokens) and len(chunks) < max_chunks:
        end = min(start + chunk_size, len(tokens))
        chunks.append(tokenizer.decode(tokens[start:end], skip_special_tokens=True))
        start += chunk_size - stride
    return chunks

print("Helper functions defined.")

In [ ]:
# Cell 4: Load and chunk Reddit posts
# This reads all 108 CSVs and chunks the post text.
# Expected: 1-5 million chunks depending on corpus size.
import pandas as pd
from transformers import AutoTokenizer
from tqdm import tqdm

tok = AutoTokenizer.from_pretrained("roberta-base")

all_posts = []
csv_files = [f for f in os.listdir(REDDIT_DIR) if f.endswith('.csv')]
print(f"Reading {len(csv_files)} CSV files...")

for fname in tqdm(csv_files, desc="Reading CSVs"):
    fpath = os.path.join(REDDIT_DIR, fname)
    try:
        df = pd.read_csv(fpath, on_bad_lines="skip",
                         usecols=lambda c: c in ["post", "body", "selftext"])
        for col in ["post", "body", "selftext"]:
            if col in df.columns:
                all_posts.extend(df[col].dropna().tolist())
                break
    except Exception as e:
        print(f"  Skipping {fname}: {e}")

print(f"Raw posts loaded: {len(all_posts):,}")

chunks = []
for post in tqdm(all_posts, desc="Chunking"):
    cleaned = clean_text(str(post))
    if not cleaned:
        continue
    chunks.extend(chunk_text(cleaned, tok))

print(f"Total chunks: {len(chunks):,}")

# Save chunks list to Drive as checkpoint
import pickle
with open(f"{BASE}/data/indexes/chunks_checkpoint.pkl", "wb") as f:
    pickle.dump(chunks, f)
print("Chunks saved to Drive checkpoint.")

In [ ]:
# Cell 5: Encode embeddings
# Uses A100 GPU. Saves progress every 100K chunks so Colab disconnects are recoverable.
import numpy as np
from sentence_transformers import SentenceTransformer
import torch

BATCH_SIZE = 256  # Safe for A100 40GB. Reduce to 64 if OOM.
SAVE_EVERY = 100_000  # Save checkpoint every N chunks

encoder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
print(f"Encoding on: {encoder.device}")
print(f"Total chunks to encode: {len(chunks):,}")
print(f"Estimated time at batch_size={BATCH_SIZE}: ~{len(chunks) // (BATCH_SIZE * 50)} minutes")

# Check if partial checkpoint exists
all_embeddings = []
start_idx = 0

if os.path.exists(CKPT_PATH):
    print(f"Resuming from checkpoint: {CKPT_PATH}")
    all_embeddings = list(np.load(CKPT_PATH))
    start_idx = len(all_embeddings)
    print(f"Resuming from chunk {start_idx:,}")

# Encode in segments
for seg_start in tqdm(range(start_idx, len(chunks), SAVE_EVERY), desc="Encoding segments"):
    seg_end = min(seg_start + SAVE_EVERY, len(chunks))
    seg = chunks[seg_start:seg_end]
    emb = encoder.encode(seg, batch_size=BATCH_SIZE, show_progress_bar=False,
                         normalize_embeddings=True, convert_to_numpy=True)
    all_embeddings.extend(emb)
    # Save checkpoint
    np.save(CKPT_PATH, np.array(all_embeddings, dtype=np.float32))
    print(f"  Checkpoint saved at {seg_end:,}/{len(chunks):,} chunks")

embeddings = np.array(all_embeddings, dtype=np.float32)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
# Cell 6: Build FAISS index and SQLite sidecar
import faiss
import sqlite3

dim = embeddings.shape[1]  # 768
n   = embeddings.shape[0]

print(f"Building FAISS index for {n:,} vectors of dim {dim}...")

if n > 100_000:
    quantizer = faiss.IndexFlatL2(dim)
    index = faiss.IndexIVFFlat(quantizer, dim, 100)
    print("Training IVFFlat index (this takes a few minutes)...")
    index.train(embeddings)
else:
    index = faiss.IndexFlatL2(dim)

index.add(embeddings)
faiss.write_index(index, INDEX_PATH)
print(f"FAISS index saved: {index.ntotal:,} vectors → {INDEX_PATH}")

# SQLite sidecar
conn = sqlite3.connect(DB_PATH)
c = conn.cursor()
c.execute("""CREATE TABLE IF NOT EXISTS chunks (
    id INTEGER PRIMARY KEY,
    text TEXT,
    emotion_label INTEGER DEFAULT -1,
    safety_score REAL DEFAULT 0.7,
    source TEXT
)""")

BATCH = 10_000
for i in tqdm(range(0, len(chunks), BATCH), desc="Writing SQLite"):
    batch = chunks[i:i+BATCH]
    c.executemany(
        "INSERT OR REPLACE INTO chunks VALUES (?,?,?,?,?)",
        [(i+j, text, -1, 0.7, "reddit") for j, text in enumerate(batch)]
    )
    conn.commit()
conn.close()
print(f"SQLite DB saved: {len(chunks):,} rows → {DB_PATH}")

In [ ]:
# Cell 7: Verify outputs
import faiss, sqlite3

idx = faiss.read_index(INDEX_PATH)
print(f"FAISS index: {idx.ntotal:,} vectors")

conn = sqlite3.connect(DB_PATH)
row_count = conn.execute("SELECT COUNT(*) FROM chunks").fetchone()[0]
sample = conn.execute("SELECT text FROM chunks LIMIT 3").fetchall()
conn.close()
print(f"SQLite rows: {row_count:,}")
print("Sample chunks:")
for i, (t,) in enumerate(sample):
    print(f"  [{i}] {t[:100]}...")

print()
print("=== BUILD COMPLETE ===")
print(f"Download these two files from Drive:")
print(f"  {INDEX_PATH}")
print(f"  {DB_PATH}")
print("Place them in your local data/indexes/ folder.")